# Ollama Cloud API key · 2.1.4

Input value=37 → llamada real de Tool → evidencia. Las celdas live consumen inferencia y requieren presupuesto activo en la campaña. No se reintenta ni se cambia de modelo. Offline solo valida las declaraciones independientes.

Qwen no estaba disponible en el catálogo cloud; Gemma4:31b fue autorizado expresamente. Este notebook congela ese modelo y no selecciona sustitutos automáticos. Qwen local se prueba en 05_ollama_gpu.


In [2]:
import agentic_systems as toolkit
assert toolkit.__version__ == '2.1.4'


In [3]:
@toolkit.tool
def certify(value: int) -> dict:
    return {'value': value, 'verified': True}
capability = toolkit.skill(name='evidence', tools=(certify,))


In [4]:
actor_framework = 'native'


In [ ]:
import os
runtime = toolkit.runtime(provider='ollama-runtime', model=os.environ['OLLAMA_MODEL'], api_key=os.environ['OLLAMA_API_KEY'], endpoint=os.environ['OLLAMA_BASE_URL'], max_tokens=512, max_turns=3, max_tool_calls=1, max_retries=0, timeout_s=90)
assert runtime.model == 'gemma4:31b', 'dev5 freezes the explicitly authorized Ollama cloud model'
print({'provider': runtime.provider, 'model': runtime.model})


In [ ]:
actor = toolkit.Agent('verification', runtime=runtime, framework=toolkit.framework(actor_framework), skills=(capability,), instructions='Call certify once with the supplied value. The Tool output is the result.', contract={'tool_expectation': {'all_of': ['certify']}, 'completion': 'when_required_tools_satisfied'}, policy={'repair': False})
result = actor.run({'value': 37}, timeout_s=120)
events = [event for node in result.walk() for event in node.tool_events]
assert result.ok and result.check_invariants().ok, result.to_dict()
assert any((e.name == 'certify' and e.ok and (e.input == {'value': 37}) for e in events))
print({'input': {'value': 37}, 'process': 'runtime + actor -> Tool event', 'output': result.data})
